In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor
from openbabel import openbabel
from openpyxl import Workbook
import time
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob

In [ ]:

sobtop_directory = result["sobtop_home"]
print(sobtop_directory)

In [ ]:



def run_sobtop(mol2_path, chg_path, top_path, itp_path):
    
    
    commands = f'7\n10\n{chg_path}\n0\n1\n2\n4\n{top_path}\n{itp_path}\n0\n'
    
    
    subprocess.run(['cd', sobtop_directory], shell=True)
    
    
    
    process = subprocess.Popen(
        ['./sobtop', mol2_path],
        cwd=sobtop_directory,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    
    stdout, stderr = process.communicate(commands)

    
    print(stdout)
    
    
    if stderr:
        print(stderr)

In [ ]:

def create_polymer_itp_top():
    for filename in os.listdir('.'):
        if filename.endswith('.pdb') and filename.startswith('polymer'):
            
            base_filename = os.path.splitext(filename)[0]
            print(base_filename)
            
            mol2_path = os.path.abspath(base_filename + '.mol2')
            chg_path = os.path.abspath(base_filename + '.chg')
            top_path = os.path.abspath(base_filename + '.top')
            itp_path = os.path.abspath(base_filename + '.itp')
            
            run_sobtop(mol2_path, chg_path, top_path, itp_path)


    
    pattern = re.compile(r"^\[ moleculetype \]\s*\n;\s*name\s+nrexcl\s*\npolymer_(\S+)\s+(\d+)", re.MULTILINE)
    
    
    for filename in os.listdir('.'):
        
        if filename.endswith('.itp') and filename.startswith('polymer'):
            
            with open(filename, 'r') as file:
                content = file.read()
    
            
            new_content = pattern.sub(r"[ moleculetype ]\n; name          nrexcl\n\1       \2", content)
    
            
            if new_content != content:
                with open(filename, 'w') as file:
                    file.write(new_content)
    
    print("All matching .itp files have been processed.")


In [ ]:
def main():
    
    df = pd.read_excel('System.xlsx')

    
    polymer_name = []
    for index, row in df.iterrows():
        
        if row['is polymer']:
            
            name = str(row['Name'])
            
            polymer_name.append(name)
    
    
    for name in polymer_name:
        
        if name in df['Name'].values:
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")
            
            create_polymer_itp_top()
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")


In [ ]:
if __name__ == '__main__':
    main()

In [ ]:

df = pd.read_excel('System.xlsx')


polymer_name = []
for index, row in df.iterrows():
    
    if row['is polymer']:
        
        name = str(row['Name'])
        
        polymer_name.append(name)

In [ ]:
polymer_name

In [ ]:

monomer_dir = 'monomer'
if not os.path.exists(monomer_dir):
    os.makedirs(monomer_dir)


for name in polymer_name:
    
    for filename in os.listdir('.'):
        if filename.startswith(f"{name}"):
            shutil.move(filename, os.path.join(monomer_dir, filename))


for name in polymer_name:
    
    for filename in os.listdir('.'):
        
        if filename.startswith(f"polymer_{name}"):
            
            base, ext = os.path.splitext(filename)
            
            new_name = f"{name}{ext}"
            
            os.rename(filename, new_name)

print("Public message removed for release.")
